# ML-Template

### Table of Contents

### [1. Exploratory Analysis](#1-exploratory-analysis)
- [1.1 Load Data](#11-load-data)
- [1.2 Plot distributions for numerical features](#12-plot-distributions-for-numerical-features)  
- [1.3 Target Variable Analysis](#13-target-variable-analysis)
- [1.4 Outlier Analysis](#14-outlier-analysis)
- [1.5 Correlation Analysis](#15-correlation-analysis)

### [2. Pre-Processing](#2-pre-processing)
- [2.1 Feature Engineering](#21-feature-engineering)
- [2.2 Data splitting](#22-data-splitting)
- [2.3 Optional PCA](#23-optional-pca)
- [2.4 Standardization](#24-standardization)
- [2.5 Data Leakage Check](#25-data-leakage-check)  
* Think about feature selection / why are features what they are  
* Feature Importance (Information Coefficient) via simple Gradient Boosting Model

### [3. Establish Baseline & Model Selection](#3-establish-baseline--model-selection)
- [3.1 Simple Baseline Model](#31-simple-baseline-model)
- [3.2 Model Selection with K-Folds CV](#32-model-selection-with-k-folds-cv)
- [3.3 Hyperparameter Tuning](#33-hyperparameter-tuning)

### [4. Model Training](#4-model-training)
- [4.1 Final Model Training on Full Training Set](#41-final-model-training-on-full-training-set)
- [4.2 Performance Analysis on K-FOLDS CV Results](#42-performance-analysis-on-k-folds-cv-results)

### [5. Model Testing](#5-model-testing)
- [5.1 Test Set Evaluation](#51-test-set-evaluation)
- [5.2 Over/Underfitting Analysis](#52-overunderfitting-analysis)
- [5.3 Model Interpretation](#53-model-interpretation)

### [6. Summary](#6-summary)
- [6.1 Model Summary](#61-model-summary)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
import warnings
warnings.filterwarnings('ignore')

## 1: Exploratory Analysis

#### 1.1 Load Data

In [ ]:
# Load your data
df = pd.read_csv('./Data/{df name in data folder}')
df.head()

In [ ]:
print("="*50)
print("DATA AUDIT")
print("="*50)

# Basic data info
print("Dataset shape:", df.shape)
print("\nData types:")
print(df.dtypes)
print("\nBasic statistics:")
print(df.describe())

# Check for duplicates
print(f"\nDuplicate rows: {df.duplicated().sum()}")

# Missing values
print("\nMissing values:")
missing_data = df.isnull().sum()
print(missing_data[missing_data > 0])

print("\n" + "="*50)
print("FEATURE PLOTS")
print("="*50)

#### 1.2 Plot distributions for numerical features

In [ ]:
print("\n" + "="*50)
print("FEATURE PLOTS")
print("="*50)

# Plot distributions for numerical features
numerical_features = df.select_dtypes(include=[np.number]).columns
n_cols = 3
n_rows = (len(numerical_features) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5*n_rows))
axes = axes.flatten() if n_rows > 1 else [axes]

for i, feature in enumerate(numerical_features):
    if i < len(axes):
        df[feature].hist(bins=30, ax=axes[i])
        axes[i].set_title(f'Distribution of {feature}')
        axes[i].set_xlabel(feature)
        axes[i].set_ylabel('Frequency')

# Remove empty subplots
for i in range(len(numerical_features), len(axes)):
    fig.delaxes(axes[i])

plt.tight_layout()
plt.show()

In [ ]:
print("\n" + "="*50)
print("SCATTER PLOTS FOR INTERESTING FEATURE COMPONENTS")
print("="*50)

# Create scatter plots for feature pairs (modify as needed)
# Example: plotting first few numerical features against each other
if len(numerical_features) >= 2:
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    axes = axes.flatten()
    
    # You can customize these pairs based on your domain knowledge
    feature_pairs = [
        (numerical_features[0], numerical_features[1]),
        (numerical_features[0], numerical_features[2]) if len(numerical_features) > 2 else (numerical_features[0], numerical_features[1]),
        (numerical_features[1], numerical_features[2]) if len(numerical_features) > 2 else (numerical_features[0], numerical_features[1]),
        (numerical_features[0], numerical_features[-1])
    ]
    
    for i, (feat1, feat2) in enumerate(feature_pairs):
        if i < 4:
            axes[i].scatter(df[feat1], df[feat2], alpha=0.6)
            axes[i].set_xlabel(feat1)
            axes[i].set_ylabel(feat2)
            axes[i].set_title(f'{feat1} vs {feat2}')
    
    plt.tight_layout()
    plt.show()

#### 1.3 Target Variable Analysis

In [ ]:
# Assuming your target variable is named 'target' - modify as needed
target_column = 'target'  # Change this to your actual target column name

print("\n" + "="*50)
print("TARGET VARIABLE ANALYSIS")
print("="*50)

if target_column in df.columns:
    print(f"Target variable: {target_column}")
    print(f"Target distribution:")
    print(df[target_column].value_counts())
    
    # Plot target distribution
    plt.figure(figsize=(8, 6))
    df[target_column].value_counts().plot(kind='bar')
    plt.title(f'Distribution of {target_column}')
    plt.xlabel(target_column)
    plt.ylabel('Count')
    plt.xticks(rotation=45)
    plt.show()
else:
    print("Please specify your target column name")


TARGET VARIABLE ANALYSIS
Please specify your target column name


#### 1.4 Outlier Analysis

In [ ]:
# Box plots for numerical features
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5*n_rows))
axes = axes.flatten() if n_rows > 1 else [axes]

print("\n" + "="*50)
print("OUTLIER ANALYSIS")
print("="*50)

for i, feature in enumerate(numerical_features):
    if i < len(axes):
        df.boxplot(column=feature, ax=axes[i])
        axes[i].set_title(f'Box Plot of {feature}')

# Remove empty subplots
for i in range(len(numerical_features), len(axes)):
    fig.delaxes(axes[i])

plt.tight_layout()
plt.show()

In [ ]:
# Statistical outlier detection (IQR method)
print("Outlier detection using IQR method:")
for feature in numerical_features:
    Q1 = df[feature].quantile(0.25)
    Q3 = df[feature].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = df[(df[feature] < lower_bound) | (df[feature] > upper_bound)]
    print(f"{feature}: {len(outliers)} outliers ({len(outliers)/len(df)*100:.2f}%)")

#### 1.5 Correlation Analysis

In [ ]:
# Correlation matrix
correlation_matrix = df[numerical_features].corr()
print("Correlation Matrix:")
print(correlation_matrix)

# Heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, 
            square=True, linewidths=0.5)
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
# High correlation pairs
high_corr_pairs = []
for i in range(len(correlation_matrix.columns)):
    for j in range(i+1, len(correlation_matrix.columns)):
        if abs(correlation_matrix.iloc[i, j]) > 0.7:  # Threshold for high correlation
            high_corr_pairs.append((correlation_matrix.columns[i], 
                                  correlation_matrix.columns[j], 
                                  correlation_matrix.iloc[i, j]))

if high_corr_pairs:
    print("\nHighly correlated feature pairs (|correlation| > 0.7):")
    for feat1, feat2, corr in high_corr_pairs:
        print(f"{feat1} - {feat2}: {corr:.3f}")
else:
    print("\nNo highly correlated feature pairs found (threshold: 0.7)")

## 2: PRE-PROCESSING

#### 2.1 Feature Engineering

#### 2.2 Data splitting

In [ ]:
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
# print(f"Training set size: {X_train.shape}")
# print(f"Test set size: {X_test.shape}")

#### 2.3 Optional PCA

#### 2.4 Standardization

In [ ]:
# scaler = StandardScaler()
# X_train_scaled = scaler.fit_transform(X_train)
# X_test_scaled = scaler.transform(X_test)

#### 2.5 Data Leakage Check

## 3: Establish Baseline & Model Selection

#### 3.1 Simple Baseline Model

In [ ]:
# Train and Test simple Baseline model here

#### 3.2 Model Selection with K-Folds CV

In [ ]:
print("\n" + "="*50)
print("MODEL SELECTION WITH K-FOLDS CV")
print("="*50)

# Set up K-Fold cross validation
k_folds = 5
kf = KFold(n_splits=k_folds, shuffle=True, random_state=42)

# Define models to test
models = {
    'Logistic Regression': LogisticRegression(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'SVM': SVC(random_state=42)
}

# Store results
cv_results = {}


MODEL SELECTION WITH K-FOLDS CV


In [ ]:
"""
print("Comparing models using K-Fold Cross Validation:")
print("-" * 60)

for name, model in models.items():
    # Perform cross validation
    cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=kf, scoring='accuracy')
    
    cv_results[name] = {
        'scores': cv_scores,
        'mean': cv_scores.mean(),
        'std': cv_scores.std()
    }
    
    print(f"{name}:")
    print(f"  CV Scores: {cv_scores}")
    print(f"  Mean CV Score: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")
    print()

# Find best model
best_model_name = max(cv_results.keys(), key=lambda x: cv_results[x]['mean'])
print(f"Best model: {best_model_name} with mean CV score: {cv_results[best_model_name]['mean']:.4f}")
"""


'\nprint("Comparing models using K-Fold Cross Validation:")\nprint("-" * 60)\n\nfor name, model in models.items():\n    # Perform cross validation\n    cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=kf, scoring=\'accuracy\')\n    \n    cv_results[name] = {\n        \'scores\': cv_scores,\n        \'mean\': cv_scores.mean(),\n        \'std\': cv_scores.std()\n    }\n    \n    print(f"{name}:")\n    print(f"  CV Scores: {cv_scores}")\n    print(f"  Mean CV Score: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")\n    print()\n\n# Find best model\nbest_model_name = max(cv_results.keys(), key=lambda x: cv_results[x][\'mean\'])\nprint(f"Best model: {best_model_name} with mean CV score: {cv_results[best_model_name][\'mean\']:.4f}")\n'

#### 3.3 Hyperparameter Tuning

In [ ]:
print("\n" + "="*50)
print("HYPERPARAMETER TUNING WITH K-FOLDS CV")
print("="*50)

# Example hyperparameter tuning for Random Forest
# Uncomment when ready to use
"""
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7, None],
    'min_samples_split': [2, 5, 10]
}

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=kf,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train_scaled, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best cross-validation score:", grid_search.best_score_)
best_model = grid_search.best_estimator_
"""


HYPERPARAMETER TUNING WITH K-FOLDS CV


'\nparam_grid = {\n    \'n_estimators\': [50, 100, 200],\n    \'max_depth\': [3, 5, 7, None],\n    \'min_samples_split\': [2, 5, 10]\n}\n\ngrid_search = GridSearchCV(\n    estimator=RandomForestClassifier(random_state=42),\n    param_grid=param_grid,\n    cv=kf,\n    scoring=\'accuracy\',\n    n_jobs=-1,\n    verbose=1\n)\n\ngrid_search.fit(X_train_scaled, y_train)\n\nprint("Best parameters:", grid_search.best_params_)\nprint("Best cross-validation score:", grid_search.best_score_)\nbest_model = grid_search.best_estimator_\n'

## 4: Model Training

#### 4.1 Final Model Training on Full Training Set

In [ ]:
print("\n" + "="*50)
print("FINAL MODEL TRAINING")
print("="*50)

# [Train final model on full training set]


FINAL MODEL TRAINING


#### 4.2 Performane Analysis on K-FOLDS CV Results

In [ ]:
print("\n" + "="*50)
print("PERFORMANCE ANALYSIS ON K-FOLDS CV RESULTS")
print("="*50)

# [Analyze averaged K-Folds CV results]


PERFORMANCE ANALYSIS ON K-FOLDS CV RESULTS


## 5: Model Testing

#### 5.1 Test Set Evaluation

In [ ]:
print("\n" + "="*50)
print("TEST SET EVALUATION")
print("="*50)

# [Evaluate model on test set]



TEST SET EVALUATION


#### 5.2 Over/Underfitting Analysis

In [ ]:
print("\n" + "="*50)
print("OVERFITTING/UNDERFITTING DIAGNOSTICS")
print("="*50)

# [Add learning curves, validation curves]


OVERFITTING/UNDERFITTING DIAGNOSTICS


#### 5.3 Model Interpretation

In [ ]:
print("\n" + "="*50)
print("MODEL INTERPRETATION")
print("="*50)

# [Add SHAP or feature importance analysis]


MODEL INTERPRETATION


## 6: Summary

#### 6.1 Model Summary

[Description of final metrics, model success, special things to mention]